In [1]:
import sys
sys.path.append('/mnt/lareaulab/faizi/Trias/src')
# load required packages
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import numpy as np
from utils import *
from transformers import BartForConditionalGeneration, PreTrainedTokenizerFast, pipeline
from transformers.utils import logging
from trias.configuration import BartConfig
from trias.tokenizer import TriasTokenizer
import re
from Bio import SeqIO
from Bio.Seq import Seq
from scipy.stats import pearsonr

# cdsFM
sys.path.append('/mnt/lareaulab/faizi/cdsFM/')
from cdsfm.decodon import AutoDeCodon
# CodoonTransformer
from transformers import AutoTokenizer, BigBirdForMaskedLM
from CodonTransformer.CodonPrediction import predict_dna_sequence
from CodonTransformer.CodonData import get_merged_seq
from CodonTransformer.CodonPrediction import validate_and_convert_organism, tokenize

# Set the aesthetic style of the plots
sns.set_theme(style="white", context="paper", palette="pastel")

logging.set_verbosity_error()

WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.1.2+cu121 with CUDA 1201 (you have 2.2.0+cu121)
    Python  3.10.13 (you have 3.10.18)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


In [3]:
## Protein expression data for zero-shot prediction
data_dir = '/mnt/lareaulab/faizi/data/'
srscu_df = pd.read_csv(data_dir + 'sRSCU.csv', index_col='species')

# CodonBERT paper
flu_vaccine = pd.read_csv(data_dir + 'downstream_data/sanofi/MLOS.csv')

# Bicknell et al paper and Gemorna paper
gfp_data = pd.read_csv(data_dir + "downstream_data/moderna/gfp_moore.csv")
luc_data = pd.read_csv(data_dir + "downstream_data/moderna/luciferase.csv")
fluc_data = pd.read_csv(data_dir + "downstream_data/gemorna/fluc.csv")
nanoluc_data = pd.read_csv(data_dir + "downstream_data/gemorna/nanoluc_leppek.csv")

In [5]:
### Load BART model and tokenizer
device = "cpu"#torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Trias
model_max_length = 2048
#checkpoint = "/mnt/lareaulab/faizi/training_output/trias_vertebrate_dec2024/checkpoint-650000/"
checkpoint = "/mnt/lareaulab/lareau/Trias/training_output/trias_mane_3/checkpoint-600000/"
vocab_file = "/mnt/lareaulab/faizi/data/vocabs/vocab.json"
tokenizer = TriasTokenizer(vocab=vocab_file, separate_vocabs=False, model_max_length=model_max_length)
model = BartForConditionalGeneration.from_pretrained(checkpoint).to(device).eval()

# cdsFM
model_decodon = AutoDeCodon.from_pretrained("goodarzilab/decodon-200M-euk")
tokenizer_decodon = model_decodon.tokenizer
tokenizer_decodon.seq_type = "dna" 
model_decodon.model.to(device)
model_decodon.model.eval()

# CodonTransformer
tokenizer_codontx = AutoTokenizer.from_pretrained("adibvafa/CodonTransformer")
model_codontx = BigBirdForMaskedLM.from_pretrained("adibvafa/CodonTransformer").to(device).eval()

flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention
flash_attn not installed, reverting to default attention


In [6]:
############ CodonTransformer sequence scoring ############
# CDS -> protein 
def cds_to_protein(cds: str):
    cds = cds.upper().replace("U", "T")
    # must be valid CDS
    if len(cds) % 3 != 0:
        return None

    protein = str(Seq(cds).translate())
    # reject internal stops
    if "*" in protein[:-1]:
        return None
    # drop terminal stop
    if protein.endswith("*"):
        protein = protein[:-1]

    return protein, cds

# Get per position codon probs
@torch.no_grad()
def get_codon_probs_for_protein(
    protein: str,
    organism: str,
    model,
    tokenizer,
    device="cpu",
):
    organism_id, _ = validate_and_convert_organism(organism)
    protein = protein.lower()

    # Build merged UNK sequence (official API)
    merged_seq = get_merged_seq(protein=protein, dna="")

    enc = tokenize(
        [{"idx": 0, "codons": merged_seq, "organism": organism_id}],
        tokenizer=tokenizer,
    ).to(device)

    out = model(**enc, return_dict=True)
    logits = out.logits  # [1, seq_len, vocab]

    # Drop [CLS] and [SEP]
    logits = logits[:, 1:-1, :]

    # Drop terminal "__unk"
    L = len(protein)
    logits = logits[:, :L, :]

    return F.softmax(logits, dim=-1)  # [1, L, vocab]

# Score a single CDS
def codon_geomean_prob(
    cds: str,
    organism: str,
    model,
    tokenizer,
    device="cpu",
):
    parsed = cds_to_protein(cds)
    if parsed is None:
        return np.nan

    protein, cds = parsed
    codons = [
        cds[i:i+3].lower()
        for i in range(0, 3 * len(protein), 3)
    ]

    probs = get_codon_probs_for_protein(
        protein,
        organism,
        model,
        tokenizer,
        device=device,
    ) 

    logps = []
    for i, (aa, codon) in enumerate(zip(protein.lower(), codons)):
        tok = f"{aa}_{codon}"
        tid = tokenizer.convert_tokens_to_ids(tok)

        if tid is None or tid < 0:
            return np.nan

        p = probs[0, i, tid].item()
        logps.append(np.log(p))

    return float(np.exp(np.mean(logps)))


# Score many CDSs
def score_sequences_codontx(
    cds_seqs,
    organism,
    model,
    tokenizer,
    device="cpu",
):
    scores = []
    for cds in cds_seqs:
        scores.append(
            codon_geomean_prob(
                cds,
                organism,
                model,
                tokenizer,
                device=device,
            )
        )
    return np.asarray(scores, dtype=float)


################# DeCodon sequence scoring ################
def _clean_to_dna(seq: str) -> str:
    s = seq.upper().replace("U", "T")
    s = "".join(c for c in s if c in "ACGT")
    return s if len(s) >= 3 else None

@torch.no_grad()
def score_sequences_decodon(
    cds_seqs,
    model_decodon,
    taxid=9606,      # Homo sapiens
    device="cpu",
):
    tokenizer = model_decodon.tokenizer
    model = model_decodon.model.to(device).eval()

    tax_token = f"<{int(taxid)}>"
    if hasattr(tokenizer, "encoder") and tax_token in tokenizer.encoder:
        tax_id = tokenizer.encoder[tax_token]
    else:
        tax_id = tokenizer.convert_tokens_to_ids(tax_token)

    if tax_id is None or tax_id < 0:
        raise ValueError(f"Taxid token {tax_token} not found in tokenizer.")

    scores = []

    for seq in cds_seqs:
        seq = _clean_to_dna(seq)
        if seq is None:
            scores.append(np.nan)
            continue

        # Mirror _auto.py behavior
        auto_seq = f"{tokenizer.cls_token}{seq}{tokenizer.sep_token}"
        enc = tokenizer(
            auto_seq,
            return_tensors="pt",
            truncation=True,
            padding=False,
            add_special_tokens=False,
        )

        input_ids = enc["input_ids"].to(device)

        # Condition on organism via CLS replacement
        input_ids[:, 0] = tax_id

        out = model(
            input_ids=input_ids,
            return_dict=True,
        )

        logits = out.logits  # [1, T, V]

        # Next-token log likelihood
        logp = F.log_softmax(logits[:, :-1, :], dim=-1)
        targets = input_ids[:, 1:]

        token_logp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)

        # Geometric mean probability per token
        score = torch.exp(token_logp.mean()).item()
        scores.append(score)

    return np.asarray(scores, dtype=float)

In [7]:
flu_seqs = flu_vaccine.CDS.tolist()
luc_seqs = luc_data["Coding sequence"].tolist()
gfp_seqs = gfp_data["Coding sequence"].tolist()
fluc_seqs = fluc_data["Coding sequence"].tolist()
nanoluc_seqs = nanoluc_data["Coding Sequence"].tolist()


datasets = {
    "nanoluc_exp_6h":  (nanoluc_seqs,    nanoluc_data["Expression (6h)"]),
    "nanoluc_exp_24":  (nanoluc_seqs,    nanoluc_data["Expression (24h)"]),
    "luc_exp":         (luc_seqs,        luc_data["AUC expression (mouse)"]),
    "fluc_exp_24h":    (fluc_seqs,       fluc_data["Expression (24h; HEK293T)"]),
    "fluc_exp_48h":    (fluc_seqs,       fluc_data["Expression (48h; HEK293T)"]),
    "gfp_exp":         (gfp_seqs,        gfp_data["AUC expression (HEK293)"]),
    "gfp_hl":          (gfp_seqs,        gfp_data["half-life (HEK293)"]),   
}

# species per dataset
species_map = {
    "gfp": "Homo sapiens",
    "luc": "Mus musculus",
    "nanoluc": "Homo sapiens",
    "fluc": "Homo sapiens",
}

In [8]:
results = {}
for key, (seqs, y_true) in datasets.items():
    sp = species_map.get(key.split("_")[0], "Homo sapiens")
    sp_id = 9606
    if sp == "Mus musculus":
        sp_id = 10090

    # compute all predictors 
    y_trias     = score_sequences(seqs, sp, model, tokenizer, device)
    y_codontx   = score_sequences_codontx(seqs, sp, model_codontx, tokenizer_codontx, device)
    y_decodon   = score_sequences_decodon(seqs, model_decodon, taxid=sp_id, device=device)
    y_gc3       = [calculate_gc3_content(s) for s in seqs]
    y_u         = [calculate_u_content(s) for s in seqs]
    y_srscu     = [calculate_average_srscu(s, sp, srscu_df) for s in seqs]
    y_mfe       = [calculate_mfe(s) for s in seqs]

    predictors = {
        "Trias": y_trias,
        "CodonTx": y_codontx,
        "DeCodon": y_decodon,
        "GC3": y_gc3,
        "U%": y_u,
        "sRSCU": y_srscu,
        "MFE": y_mfe,
    }

    results[key] = {}

    for name, y_pred in predictors.items():
        r, _ = pearsonr(y_true, y_pred)
        results[key][name] = r

model_order = ["Trias", "CodonTx", "DeCodon", "GC3", "U%", "sRSCU", "MFE"]

df = pd.DataFrame(results).T
df = df[model_order]
df.loc["Mean"] = df.abs().mean(axis=0)
df = df.round(2)

df

TypeError: calculate_average_srscu() takes 2 positional arguments but 3 were given